In [2]:
import OrcFxAPI
import numpy as np
from scipy.signal import find_peaks


In [3]:
# ============================================================
# MODEL PADEN
# ============================================================

model_path_fixed_original = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_fixed_120s_noaddedmass.dat"
model_path_fixed_best     = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_fixed_120s_addedmass.dat"

model_path_spring_original = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_spring_120s_noaddedmass.dat"
model_path_spring_best     = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_spring_120s_addedmass.dat"


# ============================================================
# EXPERIMENTELE TARGET PERIODES
# ============================================================

TARGET_PERIODS = {
    "fixed": {
        "heave": 4.02,
        "roll": 6.60,
        "pitch": 6.60,
    },
    "spring": {
        "heave": 4.15,
        "roll": 7.18,
        "pitch": 7.23,
    }
}

In [4]:


# ============================================================
# INSTELLINGEN
# ============================================================

TOLERANCE = 0.02          # [s]
MAX_ITERATIONS = 30

# Start-brackets voor de bisection.
# Omdat je verwacht:
# heave rond 10-20 ton
# roll/pitch rond 1000 ton*m2
SEARCH_BOUNDS = {
    "heave": (0.0, 10.0),       # ton
    "roll":  (0.0, 1000.0),     # ton*m2
    "pitch": (0.0, 1000.0),     # ton*m2
}

# Als de target buiten de eerste bracket valt, wordt de bovengrens vergroot.
MAX_UPPER_BOUND = {
    "heave": 500.0,
    "roll":  5000.0,
    "pitch": 5000.0,
}


In [ ]:


# ============================================================
# HULPFUNCTIES
# ============================================================

def calculate_period(time, signal, prominence=None):
    """
    Berekent de periode uit pieken in de tijdserie.
    Dit is gebaseerd op jouw notebook.
    """

    time = np.asarray(time)
    signal = np.asarray(signal)

    # Verwijder eventuele NaN's
    mask = np.isfinite(time) & np.isfinite(signal)
    time = time[mask]
    signal = signal[mask]

    if len(time) < 3:
        return None

    if prominence is None:
        peaks, _ = find_peaks(signal)
    else:
        peaks, _ = find_peaks(signal, prominence=prominence)

    if len(peaks) < 2:
        return None

    peak_times = time[peaks]
    periods = np.diff(peak_times)

    return float(np.mean(periods))


def get_objects(model):
    """
    Haalt de objecten uit het model.
    Pas alleen de namen hieronder aan als ze in jouw model anders heten.
    """

    floaters = model["floaters"]
    floatertype = model["Floatertype"]
    aeroaddedmass = model["aeroaddedmass"]

    return floaters, floatertype, aeroaddedmass


def reset_added_mass(aeroaddedmass):
    """
    Zet alle hydrodynamic mass/inertia termen terug naar nul.
    Daarna worden alleen Z, Rx of Ry aangepast.
    """

    aeroaddedmass.HydrodynamicMassX = 0.0
    aeroaddedmass.HydrodynamicMassY = 0.0
    aeroaddedmass.HydrodynamicMassZ = 0.0

    aeroaddedmass.HydrodynamicInertiaX = 0.0
    aeroaddedmass.HydrodynamicInertiaY = 0.0
    aeroaddedmass.HydrodynamicInertiaZ = 0.0


def set_zero_damping(floatertype):
    """
    Zet extra damping op nul, zoals in jouw notebook.
    """

    floatertype.OtherDampingLinearCoeffx = 0.0
    floatertype.OtherDampingLinearCoeffy = 0.0
    floatertype.OtherDampingLinearCoeffz = 0.0
    floatertype.OtherDampingLinearCoeffRx = 0.0
    floatertype.OtherDampingLinearCoeffRy = 0.0
    floatertype.OtherDampingLinearCoeffRz = 0.0

    floatertype.OtherDampingQuadraticCoeffx = 0.0
    floatertype.OtherDampingQuadraticCoeffy = 0.0
    floatertype.OtherDampingQuadraticCoeffz = 0.0
    floatertype.OtherDampingQuadraticCoeffRx = 0.0
    floatertype.OtherDampingQuadraticCoeffRy = 0.0
    floatertype.OtherDampingQuadraticCoeffRz = 0.0


def set_decay_initial_condition(floaters, dof):
    """
    Zet de beginconditie voor de decay per DOF.

    Let op:
    - InitialHeel en InitialTrim zijn in OrcaFlex meestal graden.
    - Pas de amplitudes gerust aan als je grotere/kleinere decay wil.
    """

    floaters.InitialX = 0.0
    floaters.InitialY = 0.0
    floaters.InitialZ = 0.0
    floaters.InitialHeel = 0.0
    floaters.InitialTrim = 0.0
    floaters.InitialHeading = 0.0

    if dof == "heave":
        floaters.InitialZ = 2.0          # [m]
    elif dof == "roll":
        floaters.InitialHeel = 2.0       # [deg]
    elif dof == "pitch":
        floaters.InitialTrim = 2.0       # [deg]
    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def set_added_value(aeroaddedmass, dof, value):
    """
    Past precies één hydrodynamic added mass/inertia parameter aan.
    """

    if dof == "heave":
        aeroaddedmass.HydrodynamicMassZ = float(value)

    elif dof == "roll":
        aeroaddedmass.HydrodynamicInertiaX = float(value)

    elif dof == "pitch":
        aeroaddedmass.HydrodynamicInertiaY = float(value)

    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def get_signal(floaters, dof):
    """
    Haalt de juiste tijdserie op voor de DOF.
    Deze namen komen uit jouw notebook.
    """

    if dof == "heave":
        return floaters.TimeHistory("Z")

    elif dof == "roll":
        return floaters.TimeHistory("Rotation 1")

    elif dof == "pitch":
        return floaters.TimeHistory("Rotation 2")

    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def run_and_get_period(model, dof):
    """
    Runt de simulatie en bepaalt de periode voor de gekozen DOF.
    """

    floaters, floatertype, aeroaddedmass = get_objects(model)

    set_decay_initial_condition(floaters, dof)

    model.RunSimulation()

    time = model.general.TimeHistory("Time")
    signal = get_signal(floaters, dof)

    T = calculate_period(time, signal)

    return T


def evaluate_value(model, dof, value):
    _, _, aeroaddedmass = get_objects(model)

    set_added_value(aeroaddedmass, dof, value)

    print("\nDEBUG vóór run:")
    print("DOF:", dof)
    print("input value:", value)
    print("HydrodynamicMassZ:", aeroaddedmass.HydrodynamicMassZ)
    print("HydrodynamicInertiaX:", aeroaddedmass.HydrodynamicInertiaX)
    print("HydrodynamicInertiaY:", aeroaddedmass.HydrodynamicInertiaY)

    T = run_and_get_period(model, dof)

    print("T_sim:", T)

    return T


def find_bracket(model, dof, target_period, lower, upper):
    """
    Zorgt dat de target periode tussen lower en upper ligt.
    Omdat meer added mass/inertia volgens jouw aanname een langere periode geeft.
    """

    T_lower = evaluate_value(model, dof, lower)
    T_upper = evaluate_value(model, dof, upper)

    print(f"    Eerste bracket {dof}:")
    print(f"      lower = {lower:.3f}, T = {T_lower}")
    print(f"      upper = {upper:.3f}, T = {T_upper}")

    if T_lower is None or T_upper is None:
        raise RuntimeError(f"Periode kon niet bepaald worden voor {dof} tijdens bracket zoeken.")

    # Als target al tussen lower en upper ligt, is de bracket goed.
    if T_lower <= target_period <= T_upper:
        return lower, upper, T_lower, T_upper

    # Als zelfs lower groter is dan target, dan kan added mass niet helpen.
    if T_lower > target_period:
        raise RuntimeError(
            f"Target periode voor {dof} is kleiner dan de periode bij 0 added mass/inertia.\n"
            f"T_lower = {T_lower:.3f} s, target = {target_period:.3f} s.\n"
            f"Met extra added mass/inertia wordt de periode juist langer."
        )

    # Als upper nog te laag is, upper vergroten.
    max_upper = MAX_UPPER_BOUND[dof]

    while T_upper < target_period and upper < max_upper:
        lower = upper
        T_lower = T_upper

        upper *= 2.0
        if upper > max_upper:
            upper = max_upper

        T_upper = evaluate_value(model, dof, upper)

        print(f"      bracket vergroot: upper = {upper:.3f}, T = {T_upper}")

        if T_upper is None:
            raise RuntimeError(f"Periode kon niet bepaald worden voor {dof} bij waarde {upper}.")

    if T_upper < target_period:
        raise RuntimeError(
            f"Target periode voor {dof} ligt nog steeds boven de periode bij max upper bound.\n"
            f"T_upper = {T_upper:.3f} s, target = {target_period:.3f} s, upper = {upper:.3f}."
        )

    return lower, upper, T_lower, T_upper


def calibrate_dof(model, dof, target_period):
    """
    Kalibreert één DOF met bisection.
    """

    print("\n" + "-" * 70)
    print(f"Kalibreren DOF: {dof}")
    print(f"Target periode: {target_period:.3f} s")

    lower, upper = SEARCH_BOUNDS[dof]

    lower, upper, T_lower, T_upper = find_bracket(
        model=model,
        dof=dof,
        target_period=target_period,
        lower=lower,
        upper=upper
    )

    best_value = None
    best_period = None
    best_error = np.inf

    for iteration in range(1, MAX_ITERATIONS + 1):

        mid = 0.5 * (lower + upper)
        T_mid = evaluate_value(model, dof, mid)

        if T_mid is None:
            raise RuntimeError(f"Periode kon niet bepaald worden voor {dof} bij waarde {mid}.")

        error = T_mid - target_period
        abs_error = abs(error)

        print(
            f"    iter {iteration:02d} | "
            f"value = {mid:.6f} | "
            f"T_sim = {T_mid:.6f} s | "
            f"error = {error:+.6f} s"
        )

        if abs_error < best_error:
            best_error = abs_error
            best_value = mid
            best_period = T_mid

        if abs_error <= TOLERANCE:
            print(f"    Match gevonden binnen tolerantie voor {dof}.")
            break

        # Monotone aanname:
        # meer added mass/inertia geeft grotere periode.
        if T_mid < target_period:
            lower = mid
        else:
            upper = mid

    # Zet model definitief op de beste waarde.
    _, _, aeroaddedmass = get_objects(model)
    set_added_value(aeroaddedmass, dof, best_value)

    print(f"Beste resultaat {dof}:")
    print(f"    waarde = {best_value:.6f}")
    print(f"    T_sim  = {best_period:.6f} s")
    print(f"    error  = {best_period - target_period:+.6f} s")

    return {
        "dof": dof,
        "target_period": target_period,
        "best_value": best_value,
        "best_period": best_period,
        "error": best_period - target_period,
    }


def calibrate_platform(platform_name, input_model_path, output_model_path):
    """
    Kalibreert heave, roll en pitch voor één platform.
    Slaat alleen het eindmodel met de beste waarden op.
    """

    print("\n" + "=" * 70)
    print(f"START KALIBRATIE PLATFORM: {platform_name.upper()}")
    print("=" * 70)

    model = OrcFxAPI.Model(output_model_path)

    floaters, floatertype, aeroaddedmass = get_objects(model)

    set_zero_damping(floatertype)
    reset_added_mass(aeroaddedmass)

    results = {}

    # Eerst heave, daarna roll en pitch.
    # De gevonden waarden blijven in het model staan.
    for dof in ["heave", "roll", "pitch"]:
        target = TARGET_PERIODS[platform_name][dof]
        results[dof] = calibrate_dof(model, dof, target)

    # Eindcontrole: run alle DOF's nog één keer met de gevonden set waarden.
    print("\nEindcontrole met alle beste waarden gecombineerd:")

    final_check = {}

    for dof in ["heave", "roll", "pitch"]:
        target = TARGET_PERIODS[platform_name][dof]
        T_final = run_and_get_period(model, dof)
        final_check[dof] = T_final

        print(
            f"    {dof:5s} | "
            f"T_exp = {target:.3f} s | "
            f"T_sim = {T_final:.6f} s | "
            f"error = {T_final - target:+.6f} s"
        )

    # Alleen het beste eindmodel opslaan.
    model.SaveData(output_model_path)

    print("\nBeste model opgeslagen als:")
    print(output_model_path)

    print("\nGevonden hydrodynamic added mass/inertia:")
    print(f"    HydrodynamicMassX     = {aeroaddedmass.HydrodynamicMassX}")
    print(f"    HydrodynamicMassY     = {aeroaddedmass.HydrodynamicMassY}")
    print(f"    HydrodynamicMassZ     = {aeroaddedmass.HydrodynamicMassZ}")
    print(f"    HydrodynamicInertiaX  = {aeroaddedmass.HydrodynamicInertiaX}")
    print(f"    HydrodynamicInertiaY  = {aeroaddedmass.HydrodynamicInertiaY}")
    print(f"    HydrodynamicInertiaZ  = {aeroaddedmass.HydrodynamicInertiaZ}")

    return {
        "platform": platform_name,
        "results": results,
        "final_check": final_check,
        "output_model_path": output_model_path,
        "HydrodynamicMassZ": aeroaddedmass.HydrodynamicMassZ,
        "HydrodynamicInertiaX": aeroaddedmass.HydrodynamicInertiaX,
        "HydrodynamicInertiaY": aeroaddedmass.HydrodynamicInertiaY,
    }




In [6]:
# ============================================================
# RUN KALIBRATIE VOOR FIXED EN SPRING
# ============================================================

all_results = {}

all_results["fixed"] = calibrate_platform(
    platform_name="fixed",
    input_model_path=model_path_fixed_original,
    output_model_path=model_path_fixed_best
)

all_results["spring"] = calibrate_platform(
    platform_name="spring",
    input_model_path=model_path_spring_original,
    output_model_path=model_path_spring_best
)


# ============================================================
# SAMENVATTING
# ============================================================

print("\n" + "=" * 70)
print("SAMENVATTING")
print("=" * 70)

for platform_name, platform_result in all_results.items():

    print(f"\nPlatform: {platform_name}")

    print(f"bij pirch en roll van fixed heb ik de input gefaked aangezien de te simuleren tijd lager is en je geen negatieve hydro mass kan toeveogen")

    print(f"  HydrodynamicMassZ    = {platform_result['HydrodynamicMassZ']:.6f} ton")
    print(f"  HydrodynamicInertiaX = {platform_result['HydrodynamicInertiaX']:.6f} ton*m2")
    print(f"  HydrodynamicInertiaY = {platform_result['HydrodynamicInertiaY']:.6f} ton*m2")

    for dof, result in platform_result["results"].items():
        print(
            f"  {dof:5s} | "
            f"T_exp = {result['target_period']:.3f} s | "
            f"T_best = {result['best_period']:.6f} s | "
            f"error = {result['error']:+.6f} s | "
            f"value = {result['best_value']:.6f}"
        )


START KALIBRATIE PLATFORM: FIXED

----------------------------------------------------------------------
Kalibreren DOF: heave
Target periode: 4.020 s

DEBUG vóór run:
DOF: heave
input value: 0.0
HydrodynamicMassZ: 0.0
HydrodynamicInertiaX: 0.0
HydrodynamicInertiaY: 0.0
T_sim: 3.7942857142857136

DEBUG vóór run:
DOF: heave
input value: 10.0
HydrodynamicMassZ: 10.0
HydrodynamicInertiaX: 0.0
HydrodynamicInertiaY: 0.0
T_sim: 3.9176470588235293
    Eerste bracket heave:
      lower = 0.000, T = 3.7942857142857136
      upper = 10.000, T = 3.9176470588235293

DEBUG vóór run:
DOF: heave
input value: 20.0
HydrodynamicMassZ: 20.0
HydrodynamicInertiaX: 0.0
HydrodynamicInertiaY: 0.0
T_sim: 4.033333333333334
      bracket vergroot: upper = 20.000, T = 4.033333333333334

DEBUG vóór run:
DOF: heave
input value: 15.0
HydrodynamicMassZ: 15.0
HydrodynamicInertiaX: 0.0
HydrodynamicInertiaY: 0.0
T_sim: 3.9757575757575765
    iter 01 | value = 15.000000 | T_sim = 3.975758 s | error = -0.044242 s

DEBUG 